# Gradio User Interface

In [1]:
import gradio as gr

# Closing all open ports
gr.close_all()

c:\workspace\StockPredictor\StockPredictor_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Prototype mit Funktionen

In [2]:
import pandas as pd
import kagglehub
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from pathlib import Path
import gradio as gr
from SPARQLWrapper import SPARQLWrapper, JSON

import numpy as np
from io import BytesIO
from PIL import Image

In [3]:
def updatedata(): 
    try: #Versuch den Datensatz herunterzuladen, falls es scheitert gebe das except-Statement aus
        path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating") #Download aktuelleste -version
        print("Path to dataset files:", path)
        erg = "<div style=font-size:20px;><div style=background-color:green;><center>Download successul</center></div></div>" #Download Erfolgreich
    except:
        erg = "<div style=font-size:20px;><div style=background-color:red;><center>Download failed</center></div></div>" #Download Fehlgeschlagen
    return erg

def predict(brand, days):
    df_brand = df.loc[df["Brand_Name"] == brand, ["Date", "Close", "Brand_Name"]] #Filtere Spalten für die ausgewählte Marke
    
    #Hole erste, letzte, maximale und minimale Aktienkurse
    brand_first = str(df_brand["Close"].iloc[-1])
    brand_latest = str(df_brand["Close"].iloc[0])
    max_stock = df_brand["Close"].max()
    min_stock = df_brand["Close"].min()

    #Spalten umbenennen für Autogluon-kompatibles Format
    df_brand_preproc = df_brand.rename(columns={"Date": "timestamp", "Close": "target", "Brand_Name": "item_id"})
    df_brand_preproc["item_id"] = df_brand_preproc['item_id'].astype("string")
    df_brand_preproc["timestamp"] = df_brand_preproc['timestamp'].astype("string")
    timecut = df_brand_preproc["timestamp"].str.slice(stop=10) #Schneide hh:mm:ss und timezone weg
    df_brand_preproc["timestamp"] = timecut
    df_brand_preproc["timestamp"] = pd.to_datetime(timecut) #konvertiere String in datetime64
    df_brand_reordered =  df_brand_preproc[['item_id', 'timestamp', 'target']] #Spalten neu anordnen
    df_irregular = TimeSeriesDataFrame( #Konvertiere in TimeSeriesDataFrame
        pd.DataFrame(df_brand_reordered)
    )
    df_regular = df_irregular.convert_frequency(freq="D") #Frequentierung der Daten nach D = Days
    df_filled = df_regular.fill_missing_values() #Fülle fehlende Tage (NaN) mit dem jeweils letzten bekannten Wert auf (Feiertage und Wochenenden)
    data = TimeSeriesDataFrame.from_data_frame( #Erzeuge finalen TimeSeriesDataFrame
        df = df_filled,
        id_column="item_id",
        timestamp_column="timestamp"
    )

    prediction_length = days #Setze Vorhersagezeitraum und splitte anschließend nach Trainings-/Testdaten
    train_data, test_data = data.train_test_split(prediction_length)

    predictor = TimeSeriesPredictor(prediction_length=prediction_length).fit( #Trainiere Modell Chronos Bolt Base
    train_data, presets="bolt_base",
    )

    predictions = predictor.predict(test_data) #Inferiere auf Basis der Test-Daten
    prdct_img = predictor.plot(  #Visualisiere Vorhersageergebnisse
        data=data,
        predictions=predictions,
        item_ids=data.item_ids[:2], #Maximal 2 Plots anzeigen
        max_history_length=200, #Daten der letzten 200 Tage anzeigen
    );

    #Konvertiere Plot in Bildobjekt für GradIO
    buf = BytesIO()
    prdct_img.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    image = Image.open(buf) 

    return brand_first, brand_latest, max_stock, min_stock, np.array(image) #Rückgabe der ermittelten Kurswerte + Plotbild

def knowledgegraph(user_choice):
    brands = [] #Initialisiere Liste
    data = pd.read_csv(data_path) #Datei einlesen
    arr = data["Brand_Name"].unique() #gebe jede Firma ohne dubletten an
    for i in arr: #Iteriere durch die Liste
        brands.append(i.replace(" ", "-")) #lösche bei jeder Firma die Leerzeichen und ersetze diese durch Bindestriche

    df_brands = pd.DataFrame({"brands" : brands}) #neues data frame erstellen mit den firmen
    #Manuelle Zuordnung der Wikidata-Identifier der einzelnen Unternehmen
    wikidata_elements = ["Q56276186", "Q926699", "Q3895", "Q3884", "Q312", "Q483915", "Q1046951", "Q95", "Q689141", "Q17460900", "Q7414", "Q67186598", "Q188920", "Q715583", "Q503308",
    "Q2842931", "Q478214", "Q37158", "Q182477", "Q941127", "Q9584", "Q609466", "Q868666", "Q465751", "Q96095585", "Q223127", "Q7501150", "Q128896", "Q194360",
    "Q16972754", "Q489921", "Q38076", "Q11463", "Q157062", "Q173395", "Q192314", "Q63327", "Q1141173", "Q53268", "Q1057464", "Q864407", "Q333718", "Q780442",
    "Q212405", "Q459477", "Q159433", "Q170416", "Q63335", "Q3295867", "Q2283", "Q328840", "Q504998", "Q8074134", "Q188273", "Q907311", "Q157064", "Q8093", "Q26678", "Q40993", "Q918", "Q174310", "Q30258651"]
    df_brands["wikidata"] = wikidata_elements #Füge die Wikidata-Identifier als neue Spalte hinzu

    #Suche Wikidata-ID zur vom Nutzer gewählten Marke
    company = df_brands.loc[df_brands["brands"] == user_choice.replace(" ", "-")]
    brand_identifier = company["wikidata"].loc[company.index[0]]

    #SPARQL-Abfrage
    query_start = "SELECT ?officialname ?logo ?inception ?totalassets ?revenue ?netprofit ?operatingincome ?marketcapitalization\n WHERE {"
    query_order = "wd:"+brand_identifier+" wdt:P1448 ?officialname;\n wdt:P154 ?logo;\n wdt:P571 ?inception;\n wdt:P2403 ?totalassets;\n wdt:P2139 ?revenue;\n wdt:P2295 ?netprofit;\n wdt:P3362 ?operatingincome;\n wdt:P2226 ?marketcapitalization.\n"
    query_end = "}"
    query1 = query_start+query_order+query_end

    #Sende Abfrage an Wikidata
    url = 'https://query.wikidata.org/sparql'
    user_agent = 'Visual Studio Code/1.102.0 (Windows_NT x64 10.0.19045; timucin.cicek@stud.h-da.de)' #Header mit zusätzlichen Informationen benötigt für Wikidata, siehe policy: https://foundation.wikimedia.org/wiki/Policy:Wikimedia_Foundation_User-Agent_Policy
    sparql = SPARQLWrapper(url, agent = user_agent )
    sparql.setQuery(query1)
    sparql.setReturnFormat(JSON) #Als JSON zurückgeben
    results = sparql.query().convert() #Ergebnis speichern

    #Extrahiere und formatiere Ergebnisse
    officialname = results['results']['bindings'][0]["officialname"]["value"]
    logo = results['results']['bindings'][0]["logo"]["value"]
    logo_html = "<center><img src='"+logo+"' width='100' height='100'></center></img>" #Logo als HTML-Bild anzeigen und auf 100 x 100 px verkleinern
    inception = results['results']['bindings'][0]["inception"]["value"]
    inception_short = inception[:10] #Zeige lediglich YYYY-MM-DD
    totalassets = "{:,}".format(int(results['results']['bindings'][0]["totalassets"]["value"]))+" €" #Wandel die zurückgegebenen Zahlen (Strings) in Integer und formatiere für eine bessere Leserlichkeit je 3er Stellen mit einem Komma
    revenue = "{:,}".format(int(results['results']['bindings'][0]["revenue"]["value"]))+" €"
    netprofit = "{:,}".format(int(results['results']['bindings'][0]["netprofit"]["value"]))+" €"
    operatingincome = "{:,}".format(int(results['results']['bindings'][0]["operatingincome"]["value"]))+" €"
    marketcaptl = "{:,}".format(int(results['results']['bindings'][0]["marketcapitalization"]["value"]))+" €"


    return officialname, logo_html, inception_short, totalassets, revenue, netprofit, operatingincome, marketcaptl #Rückgabe der entsprechenden Unternehmensdaten

#UI mit GradIO
with gr.Blocks() as demo:
    path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating") #Download aktuellste version
    global data_path #Als globale Variable deklarieren um in der knowledge-Function auch drauf zugreifen zu können
    data_path = path+"\World-Stock-Prices-Dataset.csv" #Pfad bauen
    stockdata = pd.read_csv(data_path) #Daten in Pandas laden
    df = pd.DataFrame(stockdata) #DataFrame mit den Daten erzeugen

    date_clean = df["Date"].str.slice(stop=10) #Timezone entfernen
    date_first = date_clean.iloc[-1] #Erstes Datum im Datensatz
    date_latest = date_clean.iloc[0] #Letztes/Aktuellsts Datum im Datensatz

    brands = df["Brand_Name"].unique().tolist() #create a list of all brands in the df

    
    title = gr.HTML("<div style=font-size:60px;'><center>Welcome to Profit Prophet</center></div>") #Titel als Html-Objekt formatieren und ausgeben
    header = gr.Image("images/ProfitProphet_Header_ChatGPT_cut.png") #Titelbild
    with gr.Row(equal_height=True): #Erster Block
        with gr.Column(scale=1):
            load = gr.Button("Update Stock")
            dataupdate = gr.HTML("<div style=font-size:20px;><div style=background-color:grey;><center>Data not updated yet</center></div></div>")
            load.click(fn=updatedata, inputs=[], outputs=dataupdate)
        with gr.Column(scale=4):
            df_presentator = df[["Date","Close","Brand_Name","Country"]]
            gr.HTML("<div style=font-size:30px;><u>10 Samples out of the Data:</u></div>")
            gr.DataFrame(df_presentator.sample(10))

    with gr.Row():
        with gr.Column(scale=1):
            nobrands = gr.Label(value="Number of Brands in Dataset: "+str(len(brands)), label="Counting Brands")
            #output = gr.Textbox(label="Output Box")
            #name = gr.Textbox(label="Name")
            
        with gr.Column(scale=1):
            latestdata = gr.Label(value="Latest Data of Stock: "+date_latest, label="Currency")

    #Konfigurationsmöglichkeiten für den User
    drpdwn = gr.Dropdown(label="Choose your Brand:", choices=brands, interactive=True) #Dropdown für Brands
    slider = gr.Slider(label="Number of Days to predict", minimum=1, maximum=50, step=1, value=3, interactive=True) #Slider für Prediction in Tagen
    prdct = gr.Button("Show me the future old man!") #Button
    
    with gr.Row(): #Zweiter Block
        with gr.Column(scale=1):
            brand_logo = gr.HTML() #Official Name & Logo + Gründungsdatum + Market Capitalization (Börsenwert)
        with gr.Column(scale=1):
            brand_titel = gr.Label(value="Official Brand Titel", label="Official Brand Titel") #Official Name & Logo + Gründungsdatum + Market Capitalization (Börsenwert)
        with gr.Column(scale=1):
            brand_inception = gr.Label(value="Inception", label="Inception") #Official Name & Logo + Gründungsdatum + Market Capitalization (Börsenwert)
        with gr.Column(scale=1):
            brand_mrktcapt = gr.Label(value="Brand Market Capitalization (Börsenwert)", label="Brand Market Capitalization (Börsenwert)") #Official Name & Logo + Gründungsdatum + Market Capitalization (Börsenwert)
    
    img1 = gr.Image()
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            start = gr.Label(value="Start Stock in Dataset", label="Start Stock")
            low = gr.Label(value="Lowest Stock", label="Historically Lowest Stock")
            
        with gr.Column(scale=1):
            latest = gr.Label(value="Latest Stock", label="Current Stock")
            high = gr.Label(value="Highest Stock", label="Historically Highest Stock")
    prdct.click(fn=predict, inputs=[drpdwn, slider], outputs=[start, latest, high, low, img1])
    
    with gr.Row(equal_height=True): #Informationen zum Unternehmen aus Wikidata
        with gr.Column(scale=1):
            assets = gr.Label(value="Total Assets", label="Total Assets (Vermögen)")
            revenue = gr.Label(value="Total Revenue", label="Total Revenue (Gesamtumsatz)")
            
        with gr.Column(scale=1):
            proft = gr.Label(value="Net Profit", label="Net Profit (Bilanzgewinn)")
            income = gr.Label(value="Operating Income", label="Operating Income (Betriebsgewinn)")

    prdct.click(fn=knowledgegraph, inputs=drpdwn, outputs=[brand_titel, brand_logo, brand_inception, assets, revenue, proft, income, brand_mrktcapt]) #Funktionslogik des Buttons "Show me the future old man!" 
    #Die Funktionslogik muss gezielt am Ende stattfinden, da nachfolgende Argumente sonst nicht mehr berücksichtig werden.

    #Mockup für den zweiten Teil des Wissensgraphen mit den Firmenspezifischen News aus den verschiedenen Portalen
    news = gr.HTML("<div style=font-size:40px;'>The Latest news of Google:</div>")
    news1 = gr.Label(value="Googles KI wird Modeberater, aber kein Ersatz für Apps", label="Spiegel")
    news2 = gr.Label(value="Google hat jetzt ein neues App-Icon", label="FAZ")
    news3 = gr.Label(value="Mexiko verklagt Google wegen >>Golf von Amerika<<", label="Bild.de")

demo.launch() #Starte App

<>:113: SyntaxWarning: invalid escape sequence '\W'
<>:113: SyntaxWarning: invalid escape sequence '\W'
C:\Users\s3phi\AppData\Local\Temp\ipykernel_41672\2283532375.py:113: SyntaxWarning: invalid escape sequence '\W'
  data_path = path+"\World-Stock-Prices-Dataset.csv" #Pfad bauen


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
